# Data used in the least-cost path model

In [1]:
import numpy as np
import pandas as pd

import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

import rasterio
from rasterio.merge import merge
from rasterio.mask import mask

import osmnx as ox

## 1) Mines

In [19]:
all_mines_df = pd.read_csv(r"..\data\output\cmr-mine-locations\all_mines_df.csv")
all_mines_df.head()

,PROP_NAME,PROP_ID,PRIMARY_COMMODITY,DEV_STAGE,MINESEARCH_ID,LATITUDE,LONGITUDE,COORDINATE_ACCURACY,STATE_PROVINCE,START_UP_YR,ACTUAL_CLOSURE_YR,EVENT_YR,Mine in Protected Area,planned_mine,MINE_OPERA,change_log,DEV_STAGE_AGGREGATED_SNL,MINE_IN_PROTECTED_AREA
0,Akonolinga,51028,iron ore,Exploration,NaN,2.91626,12.16394,Exact,South,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False
1,Akonolinga,52788,heavy mineral sands,Prefeas/Scoping,NaN,3.78500,12.13800,Exact,Centre,NaN,NaN,NaN,NaN,1,NaN,NaN,Late-stage,False
2,Bangou,65981,bauxite,Grassroots,NaN,5.24000,10.37000,Approximate,West,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False
3,Batouri,36389,gold,Exploration,118162.0,4.39820,14.40665,Exact,East,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False
4,Bibemi,50432,gold,Reserves Development,125428.0,9.38893,14.00665,Exact,North,NaN,NaN,NaN,NaN,1,NaN,NaN,Late-stage,False


In [15]:
# Stat checking
print(f"Number of mines: {len(all_mines_df)}")
print(f"Number of mines at each stage:")
print(all_mines_df["DEV_STAGE_AGGREGATED_SNL"].value_counts())

Number of mines: 44
Number of mines at each stage:
DEV_STAGE_AGGREGATED_SNL
Early-stage    17
Mine-stage     15
Late-stage     12
Name: count, dtype: int64


In [20]:
# Sort by development stage (custom order), add sequential ID as first column, and save in the same folder
stage_order = ["Early-stage", "Late-stage", "Mine-stage"]

all_mines_df = all_mines_df.copy()
all_mines_df["DEV_STAGE_AGGREGATED_SNL"] = pd.Categorical(
    all_mines_df["DEV_STAGE_AGGREGATED_SNL"],
    categories=stage_order,
    ordered=True
)

all_mines_df = all_mines_df.sort_values(
    by=["DEV_STAGE_AGGREGATED_SNL", "PROP_NAME"],
    na_position="last"
).reset_index(drop=True)

all_mines_df.insert(0, "ID", [f"cmr_{i:03d}" for i in range(1, len(all_mines_df) + 1)])

# if 'DEV_STAGE_AGGREGATED_SNL' == "Mine_stage", change to "Built"
all_mines_df["DEV_STAGE_AGGREGATED_SNL"] = all_mines_df["DEV_STAGE_AGGREGATED_SNL"].replace(
    {"Mine-stage": "Built"}
)

output_fp = Path(r"..\data\output\cmr-mine-locations\all_mines_with_id.csv")
all_mines_df.to_csv(output_fp, index=False)

print(f"Saved: {output_fp}")
all_mines_df.head()

Saved: ..\data\output\cmr-mine-locations\all_mines_with_id.csv


C:\Users\Move\AppData\Local\Temp\ipykernel_14812\2093077365.py:19: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  all_mines_df["DEV_STAGE_AGGREGATED_SNL"] = all_mines_df["DEV_STAGE_AGGREGATED_SNL"].replace(


,ID,PROP_NAME,PROP_ID,PRIMARY_COMMODITY,DEV_STAGE,MINESEARCH_ID,LATITUDE,LONGITUDE,COORDINATE_ACCURACY,STATE_PROVINCE,START_UP_YR,ACTUAL_CLOSURE_YR,EVENT_YR,Mine in Protected Area,planned_mine,MINE_OPERA,change_log,DEV_STAGE_AGGREGATED_SNL,MINE_IN_PROTECTED_AREA
0,cmr_001,Akonolinga,51028,iron ore,Exploration,NaN,2.91626,12.16394,Exact,South,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False
1,cmr_002,Bangou,65981,bauxite,Grassroots,NaN,5.24000,10.37000,Approximate,West,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False
2,cmr_003,Batouri,36389,gold,Exploration,118162.0,4.39820,14.40665,Exact,East,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False
3,cmr_004,Birsok,65980,bauxite,Target Outline,NaN,6.93000,13.07000,Exact,Adamawa,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False
4,cmr_005,CLP,90901,gold,Exploration,NaN,7.75841,13.52388,Exact,Adamawa,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False


In [21]:
# Convert mines CSV to GeoPackage with geometry
mines_gdf = gpd.GeoDataFrame(
    all_mines_df,
    geometry=gpd.points_from_xy(all_mines_df["LONGITUDE"], all_mines_df["LATITUDE"]),
    crs="EPSG:4326",
)

output_gpkg = Path(r"..\data\output\cmr-mine-locations\all_mines_with_id.gpkg")
mines_gdf.to_file(output_gpkg, driver="GPKG")

print(f"Saved: {output_gpkg}")
mines_gdf.head()

Saved: ..\data\output\cmr-mine-locations\all_mines_with_id.gpkg


,ID,PROP_NAME,PROP_ID,PRIMARY_COMMODITY,DEV_STAGE,MINESEARCH_ID,LATITUDE,LONGITUDE,COORDINATE_ACCURACY,STATE_PROVINCE,START_UP_YR,ACTUAL_CLOSURE_YR,EVENT_YR,Mine in Protected Area,planned_mine,MINE_OPERA,change_log,DEV_STAGE_AGGREGATED_SNL,MINE_IN_PROTECTED_AREA,geometry
0,cmr_001,Akonolinga,51028,iron ore,Exploration,NaN,2.91626,12.16394,Exact,South,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False,POINT (12.16394 2.91626)
1,cmr_002,Bangou,65981,bauxite,Grassroots,NaN,5.24000,10.37000,Approximate,West,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False,POINT (10.37 5.24)
2,cmr_003,Batouri,36389,gold,Exploration,118162.0,4.39820,14.40665,Exact,East,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False,POINT (14.40665 4.3982)
3,cmr_004,Birsok,65980,bauxite,Target Outline,NaN,6.93000,13.07000,Exact,Adamawa,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False,POINT (13.07 6.93)
4,cmr_005,CLP,90901,gold,Exploration,NaN,7.75841,13.52388,Exact,Adamawa,NaN,NaN,NaN,NaN,1,NaN,NaN,Early-stage,False,POINT (13.52388 7.75841)


## 2) Roads

In [2]:
roads_fp = Path(r"..\data\output\processed_roads_official\cleaned_merged_osm_heigit_liu-ver3-midterm.gpkg")
roads_gdf = gpd.read_file(roads_fp)
roads_gdf.head()

,ID,Start_pt,End_pt,Rd_length,Surface,data_src,continent,country_iso_a2,country_iso_a3,urban,...,bridge,layer,source,name:fr,name_en,name_fr,path,liu_surface,liu_idx,geometry
0,1.0,"(1103991.702615, 328198.947937)","(1103520.286835, 328787.695206)",762.730176,paved,liu,None,None,None,NaN,...,None,None,None,None,None,None,D:/GitHub/_epa_thesis/strategic-road-planning/...,paved,0.0,"MULTILINESTRING ((1103991.703 328198.948, 1103..."
1,2.0,"(1103520.286835, 328787.695206)","(1103479.510506, 327812.637704)",980.028403,paved,liu,None,None,None,NaN,...,None,None,None,None,None,None,D:/GitHub/_epa_thesis/strategic-road-planning/...,paved,1.0,"MULTILINESTRING ((1103520.287 328787.695, 1103..."
2,3.0,"(1094496.606460, 264352.420784)","(1094836.587317, 264478.208504)",364.363710,paved,liu,None,None,None,NaN,...,None,None,None,None,None,None,D:/GitHub/_epa_thesis/strategic-road-planning/...,paved,2.0,"MULTILINESTRING ((1094496.606 264352.421, 1094..."
3,4.0,"(1102316.032584, 283330.554237)","(1101453.228606, 267250.567070)",23257.130654,unpaved,liu,None,None,None,NaN,...,None,None,None,None,None,None,D:/GitHub/_epa_thesis/strategic-road-planning/...,unpaved,3.0,"MULTILINESTRING ((1102316.033 283330.554, 1102..."
4,5.0,"(1103057.420392, 327326.354672)","(1102985.552529, 327212.146541)",134.938827,paved,liu,None,None,None,NaN,...,None,None,None,None,None,None,D:/GitHub/_epa_thesis/strategic-road-planning/...,paved,4.0,"MULTILINESTRING ((1103057.42 327326.355, 11029..."


In [4]:
# Stat checking
print("CRS of roads GeoDataFrame:", roads_gdf.crs)
print(f"Number of paved and unpaved roads:")
print(roads_gdf["liu_surface"].value_counts())

CRS of roads GeoDataFrame: EPSG:3857
Number of paved and unpaved roads:
liu_surface
unpaved    271879
paved       20210
Name: count, dtype: int64


## 3) Friction map

In [18]:
friction_fp = Path(r"..\data\output\cmr-construction-cost-friction-90m\cmr_friction_90m_clipped.tif")

with rasterio.open(friction_fp) as src:
    friction_arr = src.read(1)
    friction_meta = src.meta.copy()

print(f"Shape: {friction_arr.shape}, CRS: {friction_meta['crs']}")

Shape: (15237, 10267), CRS: EPSG:4326


## 4) Spatial mask

### 4.1 Protected Areas

In [11]:
protected_areas_fp = Path(r"..\data\output\processed-protected-areas\cmr-protected-areas.tif")

with rasterio.open(protected_areas_fp) as pa_src:
    protected_areas_arr = pa_src.read(1)
    protected_areas_meta = pa_src.meta.copy()

print(f"Protected areas raster loaded: shape={protected_areas_arr.shape}, crs={protected_areas_meta['crs']}")

Protected areas raster loaded: shape=(45845, 33644), crs=EPSG:4326


## 5) Port locations

In [17]:
port_locations = {
    'Kribi': (2.9192, 9.8636),    # Deepwater port [web:22][web:1]
    'Douala': (4.0500, 9.6972),   # Main port CMDLA [web:19][web:20]
    'Limbe': (4.0236, 9.2061),    # Minor port CMLIM [web:7][web:30]
}